# Pipeline de dados

| Etapa | O que faz | Proteção |
|---|---|---|
| **1. Ingest de tracks** | parquet → `tracks` | pula se já tem linhas |
| **2. Backfill de notícias** | Alpha Vantage → `news_events` | pula semanas já cobertas |
| **3. Lock de fontes** | instala triggers, revoga escrita | pula se triggers existem |
| **4. Verificação** | contagens e cobertura | sempre executa |

In [ ]:
import time
import requests
import pandas as pd
from IPython.display import clear_output
from db import engine
from style import GREEN, setup_style
setup_style()

API = 'http://localhost:8000'

def _bar(done, total, width=30):
    pct  = done / total if total else 0
    fill = int(pct * width)
    return f"[{'█' * fill}{'░' * (width - fill)}] {done}/{total} ({pct:.0%})"

try:
    h = requests.get(f'{API}/health', timeout=5).json()
    print(f"API {h['status']}  ·  DB {h['db']}")
except Exception as e:
    print(f'API indisponível: {e}')

## 1. Ingest de tracks

Carrega `spotify_processed.parquet` na hypertable `tracks`. Pula se já tem dados.

In [ ]:
existing = pd.read_sql('SELECT count(*) AS n FROM tracks', engine).iloc[0]['n']

if existing > 0:
    print(f'Tracks já carregadas ({existing:,} linhas) — pulando.')
else:
    print('Iniciando ingest...')
    r = requests.post(f'{API}/ingest/tracks', timeout=10)
    r.raise_for_status()
    print(r.json()['message'])

    while True:
        s = requests.get(f'{API}/ingest/tracks/status', timeout=5).json()
        clear_output(wait=True)
        print(f'  rodando : {s["running"]}')
        if not s['running']:
            count = pd.read_sql('SELECT count(*) AS n FROM tracks', engine).iloc[0]['n']
            print(f'  concluído: {count:,} linhas em tracks')
            break
        time.sleep(8)

## 2. Backfill de notícias

Busca semanalmente no Alpha Vantage (entretenimento, esportes, macro). Incremental — pula semanas já no banco.

> Tier gratuito: ~25 req/dia. Progresso preservado entre reinicializações.

In [ ]:
DATE_FROM = '2023-10-18'
DATE_TO   = '2025-06-07'

cov = pd.read_sql('SELECT count(*) AS n FROM news_events', engine).iloc[0]['n']
s   = requests.get(f'{API}/ingest/news/status', timeout=5).json()
db_cov = s.get('db_coverage') or {}

full_coverage = (
    db_cov.get('earliest') is not None
    and db_cov.get('earliest', '9999') <= DATE_FROM
    and db_cov.get('latest',   '0000') >= DATE_TO
)

if full_coverage and not s.get('running'):
    print(f'Notícias já cobertas ({cov:,} artigos) — pulando.')
elif not s.get('running'):
    print(f'Iniciando backfill {DATE_FROM} → {DATE_TO}...')
    r = requests.post(f'{API}/ingest/news',
                      params={'date_from': DATE_FROM, 'date_to': DATE_TO}, timeout=10)
    r.raise_for_status()
    print(r.json()['message'])
else:
    print('Backfill já em execução — monitorando...')

In [ ]:
# progresso em tempo real — re-execute esta célula para atualizar
while True:
    s   = requests.get(f'{API}/ingest/news/status', timeout=5).json()
    cov = s.get('db_coverage') or {}
    clear_output(wait=True)
    print(f"  rodando    : {s['running']}")
    if s['running']:
        print(f"  progresso  : {_bar(s['weeks_done'], s['weeks_total'])}")
        print(f"  semana     : {s.get('week_current') or '—'}")
    print(f"  inseridos  : {s['articles_inserted']:,}")
    if cov.get('total_articles'):
        print(f"  total no BD: {cov['total_articles']:,}  ({cov['earliest']} → {cov['latest']})")
    if s.get('last_error'):
        print(f"  erro       : {s['last_error']}")
    if not s['running']:
        break
    time.sleep(15)

## 3. Lock de fontes

Instala triggers que bloqueiam UPDATE/DELETE em `tracks` e `news_events`. Pula se já instalados.

In [ ]:
trigger_existe = pd.read_sql("""
    SELECT count(*) AS n FROM pg_trigger
    WHERE  tgname = 'trg_prevent_tracks_modification'
""", engine).iloc[0]['n'] > 0

if trigger_existe:
    print('Fontes já protegidas — pulando.')
else:
    t_running = requests.get(f'{API}/ingest/tracks/status', timeout=5).json()['running']
    n_running = requests.get(f'{API}/ingest/news/status',   timeout=5).json()['running']
    if t_running or n_running:
        print('Aguarde o ingest terminar antes de fazer o lock.')
    else:
        r = requests.post(f'{API}/ingest/lock-sources', timeout=15)
        r.raise_for_status()
        print(r.json()['message'])

## 4. Verificação

In [ ]:
stats = pd.read_sql("""
    SELECT 'tracks'        AS tabela, count(*)::text AS linhas FROM tracks
    UNION ALL SELECT 'news_events',  count(*)::text FROM news_events
    UNION ALL SELECT 'song_clusters',count(*)::text FROM song_clusters
    UNION ALL SELECT 'track_lyrics', count(*)::text FROM track_lyrics
    UNION ALL SELECT 'insights',     count(*)::text FROM insights
""", engine)

tracks_range = pd.read_sql("""
    SELECT min(snapshot_date)::text AS primeiro_dia,
           max(snapshot_date)::text AS ultimo_dia,
           count(DISTINCT country)  AS paises
    FROM tracks
""", engine)

news_range = pd.read_sql("""
    SELECT min(published_at)::date::text AS primeiro_artigo,
           max(published_at)::date::text AS ultimo_artigo,
           count(DISTINCT published_at::date) AS dias_cobertos
    FROM news_events
""", engine)

locked = pd.read_sql("""
    SELECT count(*) AS n FROM pg_trigger WHERE tgname LIKE 'trg_prevent_%'
""", engine).iloc[0]['n']

print('Linhas por tabela')
print(stats.to_string(index=False))
print()
print('Cobertura de tracks')
print(tracks_range.to_string(index=False))
print()
print('Cobertura de notícias')
print(news_range.to_string(index=False))
print()
print(f'Proteção de fontes: {locked} trigger(s) instalado(s)  ({"OK" if locked >= 2 else "PENDENTE"})')